# 효과 주입 모의실험 — 실제 연구 결과 아님

**SYNTHETIC_EFFECT_INJECTION / PRELIMINARY ONLY**

관측된 설명변수·표본·잔차를 이용하되 기존 기울기를 제거하고 가상의 효과를 주입한다. 주입 효과는 1+예정회차2배에 대한 매출+1 조건부 기하평균 **0·2·5·10·20%** 변화다. 모든 업종에 같은 참 효과를 가정한다.

현재 설계와 추정 잔차에 조건부인 시연이며 실제 효과의 존재나 미래 표본의 검정력을 추정하지 않는다. 시드20261005·500회 반복을 고정하고, 결과를 본 뒤 반복을 선택하지 않는다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import os,json,hashlib
for k in ['OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','OMP_NUM_THREADS']:os.environ[k]='1'
import numpy as np
import pandas as pd
from scipy import linalg,stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
from IPython.display import display
ROOT=PROJECT_ROOT
OUT=project_path('jupyter-notebook/sandbox/reports');sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
paths=[project_path('dataset/analysis/final/revised_quarterly/revised_quarterly_panel.csv'),project_path('reports/20261006/revised_quarterly/13_effects.csv'),project_path('reports/20261006/revised_quarterly/13_model_samples.csv')]
hashes={p.relative_to(ROOT).as_posix():sha(p) for p in paths}
plan=project_json_loads((project_path('jupyter-notebook/sandbox/metadata/lab_plan.json')).read_text(encoding='utf-8'))
assert all(plan['protected_sha256'][p]==h for p,h in hashes.items())
KEY=['area_id','industry_id','quarter'];SEED=20261005;B=500;EFFECTS=[0,2,5,10,20]
panel=pd.read_csv(paths[0],dtype={k:str for k in KEY})
f=panel.loc[panel.total_sales.gt(0)&panel.night_sales.ge(0)&panel.availability_pct_centroid500.ge(100)].copy()
f=f.loc[f.groupby(['area_id','industry_id']).quarter.transform('nunique').ge(2)].sort_values(KEY).reset_index(drop=True)
assert len(f)==5015 and f.area_id.nunique()==114
f['x']=np.log1p(f.known_scheduled_start_count_centroid500);f['y']=np.log1p(f.total_sales)
ref=pd.read_csv(paths[1]).query("scenario=='primary_centroid100' and outcome=='log1p_total_sales_won'")
info=pd.read_csv(paths[2]).query("scenario=='primary_centroid100' and outcome=='log1p_total_sales_won'").iloc[0]
corr=float(info.cluster_correction);df=int(info.inference_df)
areas=sorted(f.area_id.unique());amap={a:i for i,a in enumerate(areas)}
print('실제설계',len(f),'행 /',len(areas),'상권, 가상효과',EFFECTS)

## 기준 결과 재현

상권·분기 더미를 QR분해하여 잔차화한다. 결합 모형의 CR1 보정과 자유도113을 유지하여 기존13번의 총매출 계수와 표준오차를 재현한다.

In [ ]:
models=[];check_rows=[]
for ind,d in f.groupby('industry_id',sort=True):
    d=d.reset_index(drop=True);label=d.industry.iloc[0]
    design=np.column_stack([pd.get_dummies(d.area_id,dtype=float),pd.get_dummies(d.quarter,dtype=float)])
    q,r,_=linalg.qr(design,mode='economic',pivoting=True);q=q[:,:np.linalg.matrix_rank(r)]
    x=d.x.to_numpy()-q@(q.T@d.x.to_numpy());y=d.y.to_numpy()-q@(q.T@d.y.to_numpy())
    ss=float(x@x);beta=float(x@y/ss);u=y-beta*x;ai=d.area_id.map(amap).to_numpy()
    scores=np.bincount(ai,weights=x*u,minlength=len(areas));se=float(np.sqrt(corr*(scores@scores))/ss)
    old=ref.loc[ref.industry_id.eq(ind)].iloc[0]
    assert abs(beta-old.coefficient)<1e-8 and abs(se-old.standard_error)<1e-8
    check_rows.append({'industry':label,'beta_error':abs(beta-old.coefficient),'se_error':abs(se-old.standard_error)})
    models.append({'industry_id':ind,'industry':label,'q':q,'x':x,'ss':ss,'u':u,'ai':ai})
checks=pd.DataFrame(check_rows);checks.to_csv(OUT/'03_baseline_checks.csv',index=False,encoding='utf-8-sig');display(checks)

## 상권 단위 잔차 부호와 효과 주입

같은 상권의 모든 업종·분기에 동일한 ±1 가중치를 주어 상권 내 연관을 보존한다. 상권 간 공간 상관은 재현하지 못한다. 잔차에 가중치를 준 뒤 고정효과를 다시 제거한다. 생성 기울기는 주입 기울기+잡음이며 주입0이 모의 영가설이다. 실제 관측 매출은 변경하지 않는다.

In [ ]:
rng=np.random.default_rng(SEED);weights=rng.choice([-1.,1.],size=(len(areas),B))
noise=np.empty((10,B));ses=np.empty((10,B))
for j,m in enumerate(models):
    wild=m['u'][:,None]*weights[m['ai'],:];yw=wild-m['q']@(m['q'].T@wild)
    noise[j]=m['x']@yw/m['ss'];error=yw-m['x'][:,None]*noise[j]
    score=np.zeros((len(areas),B));np.add.at(score,m['ai'],m['x'][:,None]*error)
    ses[j]=np.sqrt(corr*np.sum(score**2,axis=0))/m['ss']
assert np.isfinite(noise).all() and np.isfinite(ses).all() and (ses>0).all()
rows=[];examples=[];draws=[]
for effect in EFFECTS:
    delta=float(np.log1p(effect/100)/np.log(2));beta=delta+noise
    pval=2*stats.t.sf(np.abs(beta/ses),df)
    adj=np.column_stack([multipletests(pval[:,b],method='fdr_bh')[1] for b in range(B)])
    for j,m in enumerate(models):
        hits=int((adj[j]<.05).sum());lo,hi=proportion_confint(hits,B,method='wilson')
        rows.append({'experiment_class':'SYNTHETIC_EFFECT_INJECTION','industry_id':m['industry_id'],'industry':m['industry'],'injected_pct':effect,'injected_beta':delta,'replications':B,
            'raw_reject_pct':100*float((pval[j]<.05).mean()),'bh10_reject_pct':100*hits/B,'bh10_mc_low_pct':100*lo,'bh10_mc_high_pct':100*hi})
        examples.append({'experiment_class':'SYNTHETIC_EFFECT_INJECTION','industry':m['industry'],'injected_pct':effect,'replicate':0,'estimated_beta':beta[j,0],'standard_error':ses[j,0],'p_value':pval[j,0],'p_bh10':adj[j,0]})
        for b in range(B):draws.append({'industry_id':m['industry_id'],'injected_pct':effect,'replicate':b,'estimated_beta':beta[j,b],'standard_error':ses[j,b],'p_value':pval[j,b],'p_bh10':adj[j,b]})
power=pd.DataFrame(rows);power.to_csv(OUT/'03_power.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(examples).to_csv(OUT/'03_injected_example.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(draws).to_csv(OUT/'03_simulation_draws.csv',index=False,encoding='utf-8-sig')
display(power.pivot(index='industry',columns='injected_pct',values='bh10_reject_pct').round(1))

0%열은 이 모의 세계의 오탐 빈도, 나머지는 모든 업종에 동일한 효과를 주입한 경우의 검출 빈도다. BH10은 각 반복의10업종 검정에 적용한다. 첫 반복의 예시도 사전에 고정한 것이며 가장 작은 p값을 고르지 않았다.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'Malgun Gothic','axes.unicode_minus':False})
mat=power.pivot(index='industry',columns='injected_pct',values='bh10_reject_pct')
fig,ax=plt.subplots(figsize=(9,6));im=ax.imshow(mat.values,vmin=0,vmax=100,cmap='Blues',aspect='auto')
ax.set_xticks(range(5),[f'{x}%' for x in mat.columns]);ax.set_yticks(range(10),mat.index)
for i in range(10):
    for j in range(5):
        value=mat.iloc[i,j];ax.text(j,i,f'{value:.1f}',ha='center',va='center',color='white' if value>60 else 'black')
ax.set_xlabel('가상 주입 효과: 1+예정회차2배의 매출+1 기하평균 상대차이')
ax.set_title('실제 결과 아님 — 효과 주입 모의실험 / BH10 검출 빈도(%)')
fig.colorbar(im,ax=ax,label='모의 검출 빈도(%)');fig.tight_layout()
fig.savefig(OUT/'03_power_heatmap.png',dpi=160,bbox_inches='tight');plt.show();plt.close(fig)

## 해석과 적용 범위

가상 신호가 커질 때 이를 발견하는 빈도가 높아지는지는 분석 도구의 반응을 보여준다. 이것으로 실제 공연 효과를 발견하거나 비유의성의 원인을 확정한 것은 아니다. 추정 잔차와 유한 표본을 사용하는 wild residual 근사이며, 미래 표본 규모를 결정하는 검정력 계산으로 바로 사용할 수 없다.

실제 개선은 매출을 바꾸는 일이 아니라 독립적인 관측과 정확한 노출을 확보하는 일이다. 이 실험은 측정오차 감소나 공간 상관, 장기간 자료의 효과를 모형화하지 않는다.

### 모의 영가설의 검정 크기 확인

0%효과에서 업종별 BH10 기각 빈도는 가족 전체 FDR과 다르다. 아래는 각 반복에서 어느 업종이든 한 번 이상 BH10 기각됐는지 집계한 모의 비율이다. 원 p값의 업종별 기각률이 명목5%와 다를 수 있으므로 정확히 교정된 검정력 계산으로 읽지 않는다.

In [ ]:
null_draws=pd.DataFrame(draws).loc[lambda d:d.injected_pct.eq(0)].copy()
null_family_any_bh=100*float(null_draws.assign(hit=lambda d:d.p_bh10.lt(.05)).groupby('replicate').hit.any().mean())
calibration=pd.DataFrame([{'simulation':'synthetic global null','replications':B,'any_BH10_rejection_pct':null_family_any_bh,
 'industry_raw_rejection_min_pct':power.loc[power.injected_pct.eq(0),'raw_reject_pct'].min(),
 'industry_raw_rejection_max_pct':power.loc[power.injected_pct.eq(0),'raw_reject_pct'].max()}])
calibration.to_csv(OUT/'03_null_calibration.csv',index=False,encoding='utf-8-sig');display(calibration)

In [ ]:
assert all(sha(project_path(p))==h for p,h in hashes.items())
null=power.loc[power.injected_pct.eq(0)]
s={'status':'complete','experiment_class':'SYNTHETIC_EFFECT_INJECTION','real_world_discovery':False,'seed':SEED,'replications':B,'injected_pct_grid':EFFECTS,
 'source_sha256':hashes,'source_unchanged':True,'nobs':len(f),'clusters':len(areas),'df':df,'cluster_correction':corr,
 'baseline_max_beta_error':float(checks.beta_error.max()),'baseline_max_se_error':float(checks.se_error.max()),
 'null_bh10_rejection_min_pct':float(null.bh10_reject_pct.min()),'null_bh10_rejection_max_pct':float(null.bh10_reject_pct.max()),
 'null_raw_rejection_min_pct':float(null.raw_reject_pct.min()),'null_raw_rejection_max_pct':float(null.raw_reject_pct.max()),'null_family_any_bh_rejection_pct':null_family_any_bh,'uniform_true_effect_across_all10':True,'spatial_correlation_simulated':False,'posthoc_select_replicate':False,'power':power.to_dict('records')}
(OUT/'03_summary.json').write_text(json.dumps(s,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('완료: 실제관계 발견 아님, 본 분석 불변')